# Exploration of resampling flag variables.

There are 2 flag variables in L2_LR_SSH Unsmoothed
- `ancillary_surface_classification_flag`: ubyte containing 7 mutually exclusive flags
- `ssh_karin_2_qual`: uint containing 21 independant boolean flags

In [ ]:
%matplotlib inline

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import tol_colors as tc
import xarray as xr

import healpix_plot as hp
from healpix_plot import HealpixGrid
from healpix_resample import (
    BilinearResampler, PSFResampler, CategoricalResampler, BitmaskResampler
)
from matplotlib.colors import ListedColormap

from swot_healpix.data import SWOT
from swot_healpix.healpix_pcolormesh import healpix_pcolormesh

In [ ]:
# 15: 198m
# 10: 6.3km
level = 15

healpix_grid = HealpixGrid(level=level, indexing_scheme="nested", ellipsoid="WGS84")

track = SWOT(cycle=546, track=19, version="PGC0")

In [ ]:
swot_file = track.get_file()
swath = xr.open_dataset(swot_file, group="left", chunks=-1)

# Select interesting part

# Warrnambool (southern australia)
# swath = swath.isel(num_lines=slice(22_550, 22_800))

# Upstart bay (northern australia)
swath = swath.isel(num_lines=slice(31_500, 32_000))

swath = swath.dropna("num_pixels", subset=["longitude", "latitude"])

# Left swath, we flip to have ascending longitude
swath = swath.isel(num_pixels=slice(None, None, -1))
if track.pass_type == "descending":
    swath = swath.isel(num_lines=slice(None, None, -1))

swath

# Surface classification

The plan is to resample as float, then round to the nearest integer.

In [ ]:
classification = swath.ancillary_surface_classification_flag
class_meanings = classification.flag_meanings.split()
print(class_meanings)
classification

In [ ]:
colors = [
    tc.muted.indigo,  # open_ocean,
    tc.muted.sand,  # land,
    tc.muted.cyan,  # continental_water,
    tc.muted.teal,  # aquatic_vegetation,
    tc.muted.purple,  # continental_ice_snow,
    tc.muted.wine,  # floating_ice,
    tc.muted.rose,  # salted_basin,
    "r", # invalid
]

class_cmap = ListedColormap(colors[:-1])
class_cmap_inv = ListedColormap(colors)

### Resampler

In [ ]:
lon = swath.longitude.values.ravel()
lat = swath.latitude.values.ravel()
class_1d = classification.values.ravel()
ssh_1d = swath.ssh_karin_2.values.ravel()

resampler = CategoricalResampler(
    lon_deg=lon,
    lat_deg=lat,
    level=level
)
class_hp = resampler.resample(class_1d)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(6, 3),
    dpi=200,
    layout="constrained",
    sharex=True,
    sharey=True,
)
ax1, ax2 = axes

# Classification
im_cls = classification.plot.pcolormesh(
    x="longitude",
    y="latitude",
    ax=ax1,
    cmap=class_cmap,
    vmin=0,
    vmax=7,
    add_colorbar=False,
)
ax1.set_title("Surface class.")

# Classification (Healpix)
hp.pcolormesh(
    class_hp.cell_ids,
    np.rint(class_hp.cell_data),
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=0,
    vmax=7,
    cmap=class_cmap,
    ax=ax2,
)
cbar = fig.colorbar(im_cls, ax=axes, ticks=np.arange(0, 7) + 0.5)
cbar.set_ticklabels(class_meanings)
ax2.set_title("Surface class. (Healpix)")

ax1.set_extent((147.6, 147.8, -19.9, -19.75), crs=ccrs.PlateCarree())

for ax in axes:
    ax.coastlines(lw=0.7)
    ax.set_aspect("equal")
    ax.gridlines(draw_labels={"bottom": "x"})
ax1.gridlines(draw_labels={"left": "y"})

## Naive approach

Simply resample the classification value. This could lead to non-sensical values.
For instance open-ocean (value 0) pixels neighboring floating-ice (value 5) pixels could create values in between.

In [ ]:
lon = swath.longitude.values.ravel()
lat = swath.latitude.values.ravel()
class_1d = classification.values.ravel()
ssh_1d = swath.ssh_karin_2.values.ravel()

In [ ]:
resampler = PSFResampler(
    lon_deg=lon,
    lat_deg=lat,
    level=level,
    verbose=True,
    sigma_m=300
)

class_hp = resampler.resample(class_1d, lam=0.1)
ssh_hp = resampler.resample(ssh_1d, lam=0.1)

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(5, 5),
    dpi=200,
    layout="constrained",
    sharex=True,
    sharey=True,
)

ssh_vmin = float(swath.ssh_karin_2.quantile(0.01))
ssh_vmax = float(swath.ssh_karin_2.quantile(0.99))

# SSH
im_ssh = swath.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    ax=axes[0, 0],
    cmap="Blues_r",
    vmin=ssh_vmin,
    vmax=ssh_vmax,
    add_colorbar=False,
)
axes[0, 0].set_title("SSH")

# SSH Healpix
healpix_pcolormesh(
    ssh_hp.cell_ids,
    ssh_hp.cell_data,
    healpix_grid=healpix_grid,
    level_tile=7,
    cmap="Blues_r",
    vmin=ssh_vmin,
    vmax=ssh_vmax,
    ax=axes[0, 1],
)
fig.colorbar(im_ssh, ax=axes[0])
axes[0, 1].set_title("SSH (Healpix)")

# Classification
im_cls = classification.plot.pcolormesh(
    x="longitude",
    y="latitude",
    ax=axes[1, 0],
    cmap=class_cmap,
    vmin=0,
    vmax=7,
    add_colorbar=False,
)
axes[1, 0].set_title("Surface class.")

# Classification (Healpix)
healpix_pcolormesh(
    class_hp.cell_ids,
    np.rint(class_hp.cell_data),
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=0,
    vmax=7,
    cmap=class_cmap,
    ax=axes[1, 1],
)
cbar = fig.colorbar(im_cls, ax=axes[1], ticks=np.arange(0, 7) + 0.5)
cbar.set_ticklabels(class_meanings)
axes[1, 1].set_title("Surface class. (Healpix)")

for ax in axes.flat:
    ax.coastlines(lw=0.7)
    ax.set_aspect("equal")
    ax.gridlines()

for ax in axes[:, 0]:
    ax.gridlines(draw_labels={"left": "y"})
for ax in axes[-1]:
    ax.gridlines(draw_labels={"bottom": "x"})

In [ ]:
# Show postings to explain missing cells over land (due to layovers)

fig, ax = plt.subplots(
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(5, 2.5),
    dpi=200,
    layout="constrained",
)

healpix_pcolormesh(
    class_hp.cell_ids,
    np.rint(class_hp.cell_data),
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=0,
    vmax=7,
    cmap=class_cmap,
)

ax.scatter(
    swath.longitude,
    swath.latitude,
    marker="s",
    c="k",
    s=1,
)

ax.set_xlim(147.48, 147.64)
ax.set_ylim(-20.06, -19.96)

ax.coastlines(lw=0.7)
ax.set_aspect("equal")
ax.gridlines(draw_labels={"left": "y", "bottom": "x"})

In [ ]:
# Show coordinates along a line

i_line = 75

line_lon = swath.longitude.isel(num_lines=i_line)
line_lat = swath.latitude.isel(num_lines=i_line)

fig = plt.figure(
    layout="constrained",
    figsize=(5, 3),
    dpi=200,
)

ax1 = fig.add_subplot(121, projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(122)

classification.plot.pcolormesh(
    x="longitude", y="latitude", vmin=0, vmax=7, cmap=class_cmap, ax=ax1
)
ax1.plot(line_lon, line_lat, color="r")
ax1.set_aspect("equal")
ax1.coastlines()

ax2.axhline(0, color="k", lw=1)
ax2.plot(line_lon.diff("num_pixels"), lw=0.9, color=tc.bright.blue, label="Δlon")
ax2.plot(line_lat.diff("num_pixels"), lw=0.9, color=tc.bright.red, label="Δlat")
ax2.legend()

ax2.set_xlabel("#pixel")
ax2.set_xlim(0, len(line_lon))

## Better approach

If we only resample the value and round to nearest integer, we risk interpolating non-sensical values. Especially open_ocean (0) and floating_ice (5) risk being interpreted as any category between the two.

Idea: extract each flag as a boolean, resample that, then recombine. We may need an additional "no category" flag.

To recombine: I have seven values, I take the highest one. If it is below a threshold, mark it as invalid / uncategorized. This happens on land when there is layover, pixels gets assigned a wrong value (open-ocean).

In [ ]:
value_mask = xr.DataArray(classification.flag_values, dims=("flag"))
class_flags = value_mask == classification

In [ ]:
lon = swath.longitude.values.ravel()
lat = swath.latitude.values.ravel()
class_flags_1d = class_flags.values.reshape(class_flags.flag.size, -1)
assert class_flags_1d.shape == (len(class_meanings), lon.size)

In [ ]:
resampler = PSFResampler(lon_deg=lon, lat_deg=lat, level=level, sigma_m=300)

result = []
for flag, data in zip(class_meanings, class_flags_1d):
    resampled = resampler.resample(data, lam=0.1)
    result.append(resampled.cell_data)
    print(f"Resampled {flag}, {resampled.cg_niters} iterations, "
          f"residual {resampled.cg_residual_norms[-1]}")

class_flags_hp_psf = xr.DataArray(
    np.array(result),
    dims=("flag", "cell_ids"),
    coords={"flag": class_flags.flag, "cell_ids": resampler.cell_ids},
)

In [ ]:
resampler = BilinearResampler(lon_deg=lon, lat_deg=lat, level=level)
result = resampler.resample(class_flags_1d)

class_flags_hp_bl = xr.DataArray(
    result.cell_data,
    dims=("flag", "cell_ids"),
    coords={"flag": class_flags.flag, "cell_ids": result.cell_ids},
)

In [ ]:
class_hp_psf = class_flags_hp_psf.argmax("flag")
class_invalid_hp_psf = class_flags_hp_psf[class_hp_psf] < 0.1
class_hp_psf = class_hp_psf.where(~class_invalid_hp_psf, 7)
print(f"PSF: Found {int((class_hp_psf == 7).sum())} values with no categories")

class_hp_bl = class_flags_hp_bl.argmax("flag")
class_invalid_hp_bl = class_flags_hp_bl[class_hp_bl] < 0.1
class_hp_bl = class_hp_bl.where(~class_invalid_hp_bl, 7)
print(f"Bilin.: Found {int((class_hp_bl == 7).sum())} values with no categories")

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    sharex=True,
    sharey=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(6.6, 2.8),
    dpi=200,
    layout="constrained",
)

# Classification
im_cls = classification.plot.pcolormesh(
    x="longitude",
    y="latitude",
    ax=axes[0],
    cmap=class_cmap_inv,
    vmin=0,
    vmax=8,
    add_colorbar=False,
)
axes[0].set_title("Surface class.")

# Healpix PSF
healpix_pcolormesh(
    class_hp_psf.cell_ids,
    class_hp_psf.astype("f"),
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=0,
    vmax=8,
    cmap=class_cmap_inv,
    ax=axes[1],
)
axes[1].set_title("Healpix (PSF)")

# Healpix Bilin
healpix_pcolormesh(
    class_hp_bl.cell_ids,
    class_hp_bl.astype("f"),
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=0,
    vmax=8,
    cmap=class_cmap_inv,
    ax=axes[2],
)
axes[2].set_title("Healpix (Bilin)")

cbar = fig.colorbar(im_cls, ax=axes[2], ticks=np.arange(0, 8) + 0.5)
cbar.set_ticklabels(class_meanings + ["no_category"])

for ax in axes:
    ax.set_aspect("equal")
    ax.coastlines()

axes[0].gridlines(draw_labels={"left": "y", "bottom": "x"})
for ax in axes[1:]:
    ax.gridlines(draw_labels={"bottom": "x"})

# Quality flags

In [ ]:
qual_flags = swath.ssh_karin_2_qual
qual_meanings = qual_flags.flag_meanings.split()
print(qual_meanings)
qual_flags

In [ ]:
bit_mask = xr.DataArray(qual_flags.flag_masks, dims=("flag"), coords={"flag": qual_meanings})
qual_flags_unpack = (bit_mask & qual_flags.astype("uint")).astype("bool")

In [ ]:
# only look at variables that are not all false
variables = qual_flags_unpack.flag[(qual_flags_unpack.sum(["num_lines", "num_pixels"]) > 0).values].values

In [ ]:
fig, axes = plt.subplots(
    3,
    len(variables)//3,
    figsize=(3.5, 4),
    dpi=200,
    layout="constrained",
    sharex=True,
    sharey=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
)

for flag, ax in zip(variables, axes.flat):
    ax.pcolormesh(
        qual_flags_unpack.longitude,
        qual_flags_unpack.latitude,
        qual_flags_unpack.sel(flag=flag),
        cmap="coolwarm",
    )
    ax.annotate(
        flag,
        xy=(0, 0.5),
        xycoords="axes fraction",
        xytext=(-2, 0),
        textcoords="offset points",
        ha="right",
        va="center",
        rotation=90,
        fontsize=5
    )
    ax.set_aspect("equal")
    ax.coastlines()

In [ ]:
lon = qual_flags_unpack.longitude.values.ravel()
lat = qual_flags_unpack.latitude.values.ravel()
qual_flags_1d = qual_flags.values.ravel()

## Resampler

In [ ]:
resampler = BitmaskResampler(
    lon_deg=lon,
    lat_deg=lat,
    level=level,
    n_bits=len(qual_meanings),
)

resampled = resampler.resample(qual_flags_1d)

In [ ]:
qual_flags_hp = xr.DataArray(
    resampled.cell_data,
    dims=("cell_ids",),
    coords={"cell_ids": resampled.cell_ids}
)
bit_mask = xr.DataArray(qual_flags.flag_masks, dims=("flag",), coords={"flag": qual_meanings})
qual_flags_hp_unpack = (bit_mask & qual_flags_hp).astype("bool")

In [ ]:
flags_to_plot = [
    "suspect_large_ssh_delta",
    "suspect_large_ssh_std",
    "suspect_large_ssh_window_std",
]

fig, axes = plt.subplots(
    len(flags_to_plot),
    2,
    figsize=(4, 2 + 2 * len(flags_to_plot)),
    dpi=200,
    layout="constrained",
    sharex=True,
    sharey=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
)

xlim = [float(qual_flags.longitude.min()), float(qual_flags.longitude.max())]
ylim = [float(qual_flags.latitude.min()), float(qual_flags.latitude.max())]

for i_flag, flag in enumerate(flags_to_plot):
    ax1, ax2 = axes[i_flag, :]

    ax1.pcolormesh(
        qual_flags_unpack.longitude,
        qual_flags_unpack.latitude,
        qual_flags_unpack.sel(flag=flag),
        cmap="coolwarm",
        transform=ccrs.PlateCarree(),
    )

    healpix_pcolormesh(
        qual_flags_hp_unpack.cell_ids,
        qual_flags_hp_unpack.sel(flag=flag).astype("f"),
        healpix_grid=healpix_grid,
        level_tile=7,
        vmin=0,
        vmax=1,
        cmap="coolwarm",
        ax=ax2
    )

    ax1.annotate(
        flag,
        xy=(0, 0.5),
        xycoords="axes fraction",
        xytext=(-3, 0),
        textcoords="offset points",
        ha="right",
        va="center",
        rotation=90,
    )

    for ax in (ax1, ax2):
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect("equal")
        ax.coastlines()

axes[0, 0].set_title("SWOT", weight="semibold")
axes[0, 1].set_title("Healpix", weight="semibold")

In [ ]:
flags_to_plot = [
    "suspect_large_ssh_delta",
    "suspect_large_ssh_std",
    "suspect_large_ssh_window_std",
]

fig, axes = plt.subplots(
    len(flags_to_plot),
    2,
    figsize=(4, 2 + 2 * len(flags_to_plot)),
    dpi=200,
    layout="constrained",
    sharex=True,
    sharey=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
)

xlim = [float(qual_flags.longitude.min()), float(qual_flags.longitude.max())]
ylim = [float(qual_flags.latitude.min()), float(qual_flags.latitude.max())]

for i_flag, flag in enumerate(flags_to_plot):
    ax1, ax2 = axes[i_flag, :]

    ax2.pcolormesh(
        qual_flags.longitude,
        qual_flags.latitude,
        qual_flags.sel(flag=flag),
        cmap="coolwarm",
        transform=ccrs.PlateCarree(),
    )

    healpix_pcolormesh(
        qual_flags_hp.cell_ids,
        qual_flags_hp.sel(flag=flag).astype("f"),
        healpix_grid=healpix_grid,
        level_tile=7,
        vmin=0,
        vmax=1,
        cmap="coolwarm",
        ax=ax1
    )

    ax2.annotate(
        flag,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=-90,
    )

    for ax in (ax1, ax2):
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect("equal")
        ax.coastlines()

axes[0, 1].set_title("SWOT", weight="semibold")
axes[0, 0].set_title("Healpix", weight="semibold")